# Hyperparameter grid search for the constrained copula Wasserstein distance

Runs `copula_constrained_wasserstein_velocity_field` once per
(copula pair) x (hyperparameter combination) and records, for each run:

| column | meaning |
|---|---|
| `distance` | the estimated constrained distance, `sqrt(loss_energy)` measured on a **common evaluation grid** (see §3) |
| `loss_con` | continuity-equation violation on that same grid — the number that says whether `distance` means anything |
| `runtime_s` | wall clock of `Workspace.train()` only (the `L_fade` solve in `__init__` is excluded) |
| `iters_run` | iterations actually executed — less than `num_iterations` when the early-stop branch fired |
| `L_fade` | the training-free upper bound, read straight off `Workspace.L_fade` — the workspace already solves it for every run, so nothing extra is computed here |
| `under_fade` | whether `distance <= L_fade + upper_bound_tol`, i.e. whether the run converged at all |


In [ ]:
import itertools, json, math, time, types
from pathlib import Path

import sys
sys.path.insert(0, str(Path.cwd().parent))  # project root, so the distance_calculation_campaign imports resolve

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from distance_calculation_campaign.copula_families import load_target, MixtureCopula, KhoudrajiCopula, MirroredCopula
from distance_calculation_campaign.copula_constrained_wasserstein_velocity_field import Workspace, device

print('device:', device)
RESULTS_CSV = Path('training_grid_search_results.csv')  # appended to after every run, so an interrupted sweep is not lost

device: cuda:0


## 1. Copula pairs 

Each entry is `(name, factory)`, where `factory()` returns a fresh `(copula0, copula1)`.

In [2]:
SIGMA = 0.26

PAIRS = [
    ('gaussian0.9 -> gaussian-0.9',
     lambda: (load_target('GaussianCopula', 0.9, SIGMA),
              load_target('GaussianCopula', -0.9, SIGMA))),
    ('gaussian0.9 -> independence',
     lambda: (load_target('GaussianCopula', 0.9, SIGMA),
              load_target('IndependenceCopula', None, SIGMA))),
    ('frank25 -> clayton15',
     lambda: (load_target('FrankCopula', 25.0, SIGMA),
              load_target('ClaytonCopula', 15.0, SIGMA))),
    ('joe4 -> joe4@90',
     lambda: (load_target('JoeCopula', 4.0, SIGMA),
              MirroredCopula(load_target('JoeCopula', 4.0, SIGMA), 0))),
]

for name, factory in PAIRS:
    c0, c1 = factory()
    assert float(c0.sigma) == float(c1.sigma), f'{name}: endpoints must share sigma'
print(f'{len(PAIRS)} pairs')

4 pairs


## 2. Hyperparameter grid

`SWEEP` holds one short list per hyperparameter to vary; `BASE` holds everything held fixed.

In [ ]:
SWEEP = {
    'n_point':    [11, 16, 21, 31],
    'window':     [5000, 10000]
}

# Held fixed across the whole sweep.
BASE = dict(
    num_models=4,
    num_iterations=200000,
    penalty_weight=5,
    penalty_weight_increment=1,     
    con_tol=0.05,
    upper_bound_tol=0.02,
    lr=1e-3,                        
    jitter_grid=True,               
    energy_plateau_rel_tol=0.005,   
    features=[128, 128, 128]
)

SEED = 0            
EVAL_N_POINT = 31   

CONFIGS = [dict(zip(SWEEP, vals)) for vals in itertools.product(*SWEEP.values())]
print(f'{len(CONFIGS)} configs x {len(PAIRS)} pairs = {len(CONFIGS)*len(PAIRS)} training runs')
print('estimate the total: (runs) x (seconds per run) -- budget is 5000 GPU-h/year, max 3 concurrent jobs')

8 configs x 4 pairs = 32 training runs
estimate the total: (runs) x (seconds per run) -- budget is 5000 GPU-h/year, max 3 concurrent jobs


## 3. Scoring on a common grid

`evaluate_losses` recomputes exactly the `loss_energy` / `loss_con` block of
`Workspace.train()`, but on one fixed regular `(n, n, n)` grid with no jitter, so every
config is scored identically.


In [4]:
def evaluate_losses(module, target0, target1, n_point=EVAL_N_POINT):
    """(loss_energy, loss_con) for a trained module on a fixed regular grid.

    Mirrors the loss computation in `Workspace.train()`. No `torch.no_grad()`:
    `module.d_model` goes through `torch.func.jacrev` and needs autograd live.
    """
    t = torch.linspace(0, 1, n_point).double().to(device)
    x = torch.linspace(-1, 1, n_point).double().to(device)
    state = torch.cartesian_prod(t, x, x)
    t_list, x_list = state[..., 0], state[..., 1:]

    logf0, logf1 = target0.log_density(x_list), target1.log_density(x_list)
    dx_logden0, dx_logden1 = target0.dx_logden(x_list), target1.dx_logden(x_list)

    logrhonn, vec, C = module(state)
    logrhonn, C = logrhonn.squeeze(), C.squeeze()
    dtlogrhonn, dxlogrhonn, dvec = module.d_model(state)

    logrho = (1-t_list)*logf0 + t_list*logf1 + t_list*(1-t_list)*logrhonn
    rho = torch.exp(logrho)
    dt_logrho = -logf0 + logf1 + (1-2*t_list)*logrhonn + t_list*(1-t_list)*dtlogrhonn
    dx_logrho = ((1-t_list).unsqueeze(-1)*dx_logden0 + t_list.unsqueeze(-1)*dx_logden1
                 + (t_list*(1-t_list)).unsqueeze(-1)*dxlogrhonn)

    rho_reshape = rho.reshape((n_point, n_point, n_point))
    f_reshape = rho_reshape / (rho_reshape.mean([1, 2]) * 2**2).unsqueeze(-1).unsqueeze(-1).repeat(1, n_point, n_point)
    f = f_reshape.reshape(rho.shape)

    loss_con = torch.abs((dt_logrho + (dx_logrho*vec).sum(1) + dvec + C)*f).mean() * 2**2

    f1 = f_reshape.mean(2) * 2
    f2 = f_reshape.mean(1) * 2
    f1 = f1.unsqueeze(-1).repeat(1, 1, n_point).reshape(f.unsqueeze(-1).shape)
    f2 = f2.unsqueeze(1).repeat(1, n_point, 1).reshape(f.unsqueeze(-1).shape)
    fv = f.unsqueeze(-1) * vec
    fv_1 = fv[..., 0].reshape((n_point,)*3).mean(2).unsqueeze(-1).repeat(1, 1, n_point).reshape(f.unsqueeze(-1).shape) * 2
    fv_2 = fv[..., 1].reshape((n_point,)*3).mean(1).unsqueeze(1).repeat(1, n_point, 1).reshape(f.unsqueeze(-1).shape) * 2
    w = torch.cat([f1, f2], dim=1)*vec - torch.cat([fv_1, fv_2], dim=1)
    loss_energy = ((w**2).sum(1)*f).mean() * 2**2

    return float(loss_energy.detach()), float(loss_con.detach())

## 4. One run

In [5]:
def run_one(pair_name, factory, config, seed=SEED, verbose=False):
    """Train one (pair, config) and return a flat result row."""
    torch.manual_seed(seed)
    c0, c1 = factory()
    args = types.SimpleNamespace(ndim=2, verbose=verbose, **{**BASE, **config})
    args.features = list(args.features)   # the MLP builders do [x] + args.features + [y]

    ws = Workspace(args, c0, c1)          # __init__ also solves the fade bound -> ws.L_fade
    t0 = time.perf_counter()
    ws.train()                            # Ctrl-C returns the partially-trained model
    runtime = time.perf_counter() - t0

    energy, con = evaluate_losses(ws.get_model(), ws.target0, ws.target1)
    distance = math.sqrt(max(energy, 0.0))

    # ws.L_fade is the training-free upper bound the workspace computes for every run
    # (None if that solve failed). A distance above it means the run did not converge --
    # the linear-mixture path, which needs no training at all, is already cheaper.
    under_fade = None if ws.L_fade is None else distance <= ws.L_fade + args.upper_bound_tol

    row = {
        'pair': pair_name,
        **{k: (json.dumps(v) if isinstance(v, list) else v) for k, v in config.items()},
        'distance': distance,
        'loss_con': con,
        'loss_energy': energy,
        'runtime_s': runtime,
        'iters_run': ws.iter,
        'early_stopped': ws.iter <= args.num_iterations,
        'penalty_weight_final': args.penalty_weight,
        'con_ok': con < BASE['con_tol'],
        'L_fade': ws.L_fade,
        'under_fade': under_fade,
        'seed': seed,
        'config_key': json.dumps(config, sort_keys=True, default=str),
    }
    del ws
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return row

## 5. The sweep

Appends each row to `RESULTS_CSV` as soon as it finishes and skips `(pair, config_key)`
combinations already present, so the cell can be re-run after an interruption, a kernel
restart, or a hit `walltime` and it will pick up where it stopped.

Delete `training_grid_search_results.csv` to start over.

In [6]:
done = set()
if RESULTS_CSV.exists():
    prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(prev['pair'], prev['config_key']))
    print(f'{len(done)} runs already recorded, skipping those')

total = len(PAIRS) * len(CONFIGS)
k = 0
for (pair_name, factory), config in itertools.product(PAIRS, CONFIGS):
    k += 1
    key = (pair_name, json.dumps(config, sort_keys=True, default=str))
    if key in done:
        continue
    print(f'[{k}/{total}] {pair_name} | {config}', flush=True)
    row = run_one(pair_name, factory, config)
    print(f"    distance={row['distance']:.4f}  loss_con={row['loss_con']:.3e}  "
          f"{row['runtime_s']:.0f}s  iters={row['iters_run']}  under_fade={row['under_fade']}", flush=True)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=not RESULTS_CSV.exists(), index=False)
    done.add(key)
print('done')

[1/32] gaussian0.9 -> gaussian-0.9 | {'n_point': 11, 'window': 5000}


/hkfs/home/haicore/ior/fi1281/copula_fade_wasserstein_distance_script.py:914: UserWarning: fade_distance: both copulas have negligible density over 43% of the domain, so the fade path spends much of its time moving mass through near-empty regions. L is still a valid finite length, but a large value here is real -- it is what the linear-mixture path genuinely costs, not a numerical artefact.
  warnings.warn(f"fade_distance: {msg}. L is still a valid finite "


    distance=0.4953  loss_con=2.160e-02  1203s  iters=29999  under_fade=True


[2/32] gaussian0.9 -> gaussian-0.9 | {'n_point': 11, 'window': 10000}


/hkfs/home/haicore/ior/fi1281/copula_fade_wasserstein_distance_script.py:914: UserWarning: fade_distance: both copulas have negligible density over 43% of the domain, so the fade path spends much of its time moving mass through near-empty regions. L is still a valid finite length, but a large value here is real -- it is what the linear-mixture path genuinely costs, not a numerical artefact.
  warnings.warn(f"fade_distance: {msg}. L is still a valid finite "


...

[31/32] joe4 -> joe4@90 | {'n_point': 31, 'window': 5000}


    distance=0.4361  loss_con=1.994e-02  1683s  iters=14999  under_fade=True


[32/32] joe4 -> joe4@90 | {'n_point': 31, 'window': 10000}


    distance=0.4337  loss_con=1.619e-02  3363s  iters=29999  under_fade=True


done


## 6. Results


In [7]:
df = pd.read_csv(RESULTS_CSV)
df['gap_to_fade'] = df['L_fade'] - df['distance']   # positive = beat the training-free path
df.sort_values(['pair', 'distance'])

,pair,n_point,window,distance,loss_con,loss_energy,runtime_s,iters_run,early_stopped,penalty_weight_final,con_ok,L_fade,under_fade,seed,config_key,gap_to_fade
19,frank25 -> clayton15,16,10000,0.032128,0.017330,0.001032,1987.431031,49999,True,6,True,0.024666,True,0,"{""n_point"": 16, ""window"": 10000}",-0.007462
23,frank25 -> clayton15,31,10000,0.032272,0.018899,0.001041,3327.155165,29999,True,6,True,0.024666,True,0,"{""n_point"": 31, ""window"": 10000}",-0.007606
22,frank25 -> clayton15,31,5000,0.032473,0.016418,0.001054,2773.817491,24999,True,6,True,0.024666,True,0,"{""n_point"": 31, ""window"": 5000}",-0.007807
18,frank25 -> clayton15,16,5000,0.032769,0.038607,0.001074,1305.098039,34999,True,7,True,0.024666,True,0,"{""n_point"": 16, ""window"": 5000}",-0.008103
20,frank25 -> clayton15,21,5000,0.032815,0.028284,0.001077,908.859749,19999,True,6,True,0.024666,True,0,"{""n_point"": 21, ""window"": 5000}",-0.008149
21,frank25 -> clayton15,21,10000,0.033414,0.019312,0.001117,1361.544696,29999,True,6,True,0.024666,True,0,"{""n_point"": 21, ""window"": 10000}",-0.008748
16,frank25 -> clayton15,11,5000,0.033539,0.029012,0.001125,1445.355043,39999,True,8,True,0.024666,True,0,"{""n_point"": 11, ""window"": 5000}",-0.008873
17,frank25 -> clayton15,11,10000,0.034185,0.031095,0.001169,1804.160225,49999,True,6,True,0.024666,True,0,"{""n_point"": 11, ""window"": 10000}",-0.009519
0,gaussian0.9 -> gaussian-0.9,11,5000,0.495283,0.021598,0.245305,1202.747009,29999,True,6,True,0.633184,True,0,"{""n_point"": 11, ""window"": 5000}",0.137901
7,gaussian0.9 -> gaussian-0.9,31,10000,0.495607,0.019342,0.245626,2203.344066,19999,True,5,True,0.633184,True,0,"{""n_point"": 31, ""window"": 10000}",0.137577


In [8]:
# Best (smallest valid distance) config per pair
valid = df[df['con_ok'] & df['under_fade'].fillna(False)]
print('valid runs:', len(valid), 'of', len(df))
valid.loc[valid.groupby('pair')['distance'].idxmin()] if len(valid) else 'no valid runs yet'

valid runs: 32 of 32


,pair,n_point,window,distance,loss_con,loss_energy,runtime_s,iters_run,early_stopped,penalty_weight_final,con_ok,L_fade,under_fade,seed,config_key,gap_to_fade
19,frank25 -> clayton15,16,10000,0.032128,0.017330,0.001032,1987.431031,49999,True,6,True,0.024666,True,0,"{""n_point"": 16, ""window"": 10000}",-0.007462
0,gaussian0.9 -> gaussian-0.9,11,5000,0.495283,0.021598,0.245305,1202.747009,29999,True,6,True,0.633184,True,0,"{""n_point"": 11, ""window"": 5000}",0.137901
13,gaussian0.9 -> independence,21,10000,0.246645,0.006232,0.060834,5101.207184,119999,True,5,True,0.269088,True,0,"{""n_point"": 21, ""window"": 10000}",0.022443
29,joe4 -> joe4@90,21,10000,0.433356,0.014862,0.187798,1422.085874,29999,True,5,True,0.498665,True,0,"{""n_point"": 21, ""window"": 10000}",0.065309


In [9]:
# Marginal effect of each swept hyperparameter, averaged over the others
for h in SWEEP:
    print(f'--- {h} ---')
    print(df.groupby(h)[['distance', 'loss_con', 'runtime_s']].mean().round(4), '\n')

--- n_point ---
         distance  loss_con  runtime_s
n_point                               
11         0.3071    0.0210  1713.9257
16         0.3038    0.0187  1725.0352
21         0.3036    0.0161  1905.4310
31         0.3033    0.0152  3854.6391 

--- window ---
        distance  loss_con  runtime_s
window                               
5000      0.3052    0.0185  1966.9246
10000     0.3037    0.0170  2632.5910 



## 7. Conclusion

**Chosen discretization:** `n_point=16`, `window=10000`.

All 32 runs are valid (`con_ok` and `under_fade`), and the discretization moves the distance very little: the four `n_point` means span only 0.3033–0.3071.

- **`n_point`**: 11 is too coarse, it is worst on average in both distance and `loss_con`. From 16 upwards the distance is flat (0.3038 / 0.3036 / 0.3033). `loss_con` keeps improving slightly (0.0187 → 0.0152), but runtime doubles going from 21 to 31 (1905 s → 3855 s). 16 was chosen as the cheapest grid that is already on the plateau. 21 is about 10% slower and only marginally better.
- **`window`**: 10000 beats 5000 on both distance (0.3037 vs 0.3052) and `loss_con` (0.0170 vs 0.0185), at ~34% more runtime, because a longer window makes the early stopping check less frequent, so runs stop later and less prematurely.